In [ ]:
library(STged)
library(DOTr)
library(ggplot2)
library(SeuratObject)
library(Seurat)
library(presto)
library(dplyr)
library(SeuratDisk)
library(Matrix)

In [ ]:
python_env <- "/net/data.isilon/ag-saez/yliu/SOFTWARE/.miniconda3/envs/multi/bin/python"
reticulate::use_python(python_env, required = TRUE)
reticulate::py_config()
anndata <- reticulate::import("anndata")
np <- reticulate::import("numpy")
sq <- reticulate::import("squidpy")

In [ ]:
sc = readRDS('data/spatial/processed_data/scRNA.rds')
sc <- subset(sc, subset = nFeature_RNA > 200 & nFeature_RNA < 6000 & 
                                  nCount_RNA > 500 & nCount_RNA < 50000)
st <-readRDS('data/spatial/processed_data/spatial.rds')

In [ ]:
donor = c('BCLL-8-T','BCLL-9-T','BCLL-10-T','BCLL-11-T','BCLL-12-T','BCLL-13-T')
prefix = c('c28w2r_7jne4i_','qvwc8t_2vsr67_','esvq52_nluss5_','exvyh1_66caqq_','p7hv1g_tjgmyj_','gcyl7c_cec61b_')
donor <-'BCLL-8-T'

In [ ]:
coor <- read.csv("data/spatial/coordinates.csv", header = TRUE, stringsAsFactors = FALSE)

In [ ]:
samples <- rownames(sc@meta.data)[!sc$annotation_level_1 %in% c("preTC", "preBC")]
sc_counts <- GetAssayData(sc, assay = "RNA", slot = "counts")[ ,samples, drop = FALSE]
sc_counts <- as.matrix(sc_counts)

label <- droplevels(sc$annotation_level_1[samples])

samples <- rownames(st@meta.data)[st$donor_id == donor]
st_counts <- GetAssayData(st, assay = "Spatial", slot = 'counts')
st_counts <- as.matrix(st_counts[, samples, drop = FALSE])

slice <- 'c28w2r_7jne4i'
xy <- coor[startsWith(coor$X, slice), ]
xy <- xy[, c("row", "col")]
xy <- as.matrix(xy)
rownames(xy) <- NULL
colnames(xy) <- NULL


In [ ]:
dot_file <- paste0('data/spatial/test/', donor, '.rds')
dot <- readRDS(dot_file)
beta <- dot@weights / rowSums(dot@weights)
beta <- beta / rowSums(beta)

In [ ]:
clean.only = FALSE
depthscale  = 1e6
datax = data_process(sc_exp = sc_counts,   sc_label = label, 
                     spot_exp = st_counts,  spot_loc = xy,
                     depthscale = depthscale,  gene_det_in_min_cells_per = 0.01, 
                     expression_threshold = 0,
                     nUMI =  100, verbose = FALSE, clean.only = clean.only)

cat("Construct spatial correlation", "\n")
L.mat <- dis_weight(spot_loc = datax$spot_loc, spot_exp = datax$spot_exp, k = 4,
                    method = "Square", coord_type = "grid" )

cat("Construct reference gene matrix", "\n")
ref_exp = create_group_exp(sc_exp = datax$sc_exp, sc_label = datax$sc_label)

#the corresponding cell type proportion
beta =  beta[colnames(datax$spot_exp),]
cat("Run the STged", "\n")
stged = MUR.STged(srt_exp = datax$spot_exp, ref_exp = ref_exp, beta.type = beta,  W = L.mat$weight_adj, lambda1 = NULL, lambda2 = NULL,  cutoff = 0.05, epsilon = 1e-5,  
                      maxiter = 100)

In [ ]:
saveRDS(datax, file = 'data/spatial/test/datax.rds')
saveRDS(stged, file = 'data/spatial/test/STged.rds')

In [ ]:
ctexp  = stged$F_list
beta.type = stged$beta
colnames(beta.type)

In [ ]:
cell_type = colnames(beta.type)
ind_spot =datax$spot_exp>0

ind = beta.type[,3]>0.05

st_exp_raw = ctexp[[3]]


colnames(st_exp_raw) = rownames(beta.type)

rownames(st_exp_raw)=  gsub("(^|[[:space:]])([[:alpha:]])", "\\1\\U\\2",  
                            tolower(rownames(st_exp_raw )), perl = TRUE)

spot_loc_raw  = datax$spot_loc[rownames(beta.type) ,1:2]
spot_loc_raw$x = as.numeric(spot_loc_raw$x)
spot_loc_raw$y = as.numeric(spot_loc_raw$y)

spot_loc = spot_loc_raw[ind,]

In [ ]:
col_df <- data.frame(cell_types = c(cell_type,"Others"),
                     col_vector = c(col_vector[1:length(cell_type)],"gray"))

In [ ]:
beta = beta.type[ind ,]
cell_type = colnames(beta)
target  = j = cell_type[3]
target

In [ ]:
st_exp  = st_exp_raw[,ind ]
dim(st_exp)

In [ ]:
Y_input = t(cleanCounts (st_exp, clean.only =TRUE) )

dim(Y_input)

In [ ]:
Y_scaled <- scale(Y_input)
mHAG_est <- mHAG_func(X_scaled = X_input,
                        Y_scaled = Y_scaled, 
                        alpha = 0.5, 
                        num_cores = 10)
saveRDS(mHAG_est, file = 'data/spatial/test/ctHVG.rds')

In [ ]:
mHAG_res = mHAG_module(mHAG_est)

mHVG <- mHAG_res$mHVG

cat("number of mHVGs=",  length(mHVG), "\n")

In [ ]:
apply(mHAG_res$ctHVG,2,sum, na.rm = TRUE)